# Project 3 — Early Warning System for Sepsis (PhysioNet 2019)
## Notebook 03 — Grouped CV Modeling + OOF Predictions (for Calibration & Policy)

**Goal**
Train models using **patient-grouped** cross-validation and generate **out-of-fold (OOF)** probability predictions.
These OOF predictions are the only correct input for Notebook 04 (calibration + alert policy).

**Input**
- `results/features_02b.parquet`

**Outputs**
- `results/oof_predictions.parquet` (patient_id, ICULOS, y, p_oof, fold_id, model_name)
- `results/model_metrics_cv.json` (AUROC/AUPRC per fold + mean/std)
- (optional) `results/final_model.pkl` (trained on all data)

**Key rule**
All splits must be grouped by `patient_id`.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

pd.set_option("display.max_columns", 200)

# -------------------------
# Paths (EDIT THIS ONCE)
# -------------------------
PROJECT_ROOT = Path(".").resolve()
DATA_ROOT = PROJECT_ROOT  # set this if your results live elsewhere

RESULTS_DIR = DATA_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IN_FEATS = RESULTS_DIR / "features_02b.parquet"
OUT_OOF = RESULTS_DIR / "oof_predictions.parquet"
OUT_METRICS = RESULTS_DIR / "model_metrics_cv.json"

print("Input:", IN_FEATS, "exists:", IN_FEATS.exists())


### 1) Load engineered features
We keep:
- `patient_id`, `ICULOS` as identifiers (needed later for policy & lead-time)
- `y_within_H` as the target


In [ ]:
df = pd.read_parquet(IN_FEATS)
df = df.sort_values(["patient_id", "ICULOS"]).reset_index(drop=True)

required = {"patient_id", "ICULOS", "y_within_H"}
missing_req = required - set(df.columns)
assert not missing_req, f"Missing required columns: {missing_req}"

print("Rows:", len(df), "Patients:", df["patient_id"].nunique(), "Cols:", df.shape[1])
print("Target prevalence (row-level):", float(df["y_within_H"].mean()))
df.head()


### 2) Define X / y / groups
We exclude identifiers and target from X.


In [ ]:
ID_COLS = ["patient_id", "ICULOS"]
TARGET_COL = "y_within_H"

feature_cols = [c for c in df.columns if c not in ID_COLS + [TARGET_COL]]
X = df[feature_cols]
y = df[TARGET_COL].astype(int).values
groups = df["patient_id"].values

print("n_features:", len(feature_cols))
print("Example features:", feature_cols[:10])


### 3) Helpers: grouped CV + OOF predictions
We compute AUROC & AUPRC per fold and save OOF probabilities.


In [ ]:
def make_sample_weight(y_fold: np.ndarray) -> np.ndarray:
    # Simple inverse-frequency weights (helps imbalance)
    n_pos = (y_fold == 1).sum()
    n_neg = (y_fold == 0).sum()
    if n_pos == 0:
        return np.ones_like(y_fold, dtype=float)
    w_pos = n_neg / max(n_pos, 1)
    w = np.ones_like(y_fold, dtype=float)
    w[y_fold == 1] = w_pos
    return w

def run_group_cv_oof(model, X: pd.DataFrame, y: np.ndarray, groups: np.ndarray, n_splits=5, model_name="model"):
    """
    Patient-grouped CV with true out-of-fold (OOF) probabilities.

    Notes:
    - Uses sklearn.clone() so each fold gets a fresh estimator.
    - Passes sample weights robustly:
        * for Pipelines with final step named 'clf', uses clf__sample_weight
        * otherwise uses sample_weight if supported
    """
    gkf = GroupKFold(n_splits=n_splits)

    oof = np.full(shape=len(y), fill_value=np.nan, dtype=float)
    fold_ids = np.full(shape=len(y), fill_value=-1, dtype=int)

    fold_metrics = []

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups)):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        sw = make_sample_weight(y_tr)

        m = clone(model)

        # Fit (try best-effort weighted fit)
        fit_ok = False
        try:
            m.fit(X_tr, y_tr, clf__sample_weight=sw)  # Pipeline case (last step 'clf')
            fit_ok = True
        except TypeError:
            pass
        if not fit_ok:
            try:
                m.fit(X_tr, y_tr, sample_weight=sw)     # Estimator supports sample_weight directly
                fit_ok = True
            except TypeError:
                m.fit(X_tr, y_tr)                       # Unweighted fallback

        # Predict probability for positive class
        if hasattr(m, "predict_proba"):
            p = m.predict_proba(X_va)[:, 1]
        else:
            s = m.decision_function(X_va)
            p = 1 / (1 + np.exp(-s))

        oof[va_idx] = p
        fold_ids[va_idx] = fold

        auroc = roc_auc_score(y_va, p) if len(np.unique(y_va)) > 1 else np.nan
        auprc = average_precision_score(y_va, p) if len(np.unique(y_va)) > 1 else np.nan

        fold_metrics.append({
            "model": model_name,
            "fold": int(fold),
            "auroc": float(auroc),
            "auprc": float(auprc),
            "n_val": int(len(va_idx)),
            "pos_val": int(y_va.sum())
        })

        print(f"[{model_name}] Fold {fold}: AUROC={auroc:.4f} AUPRC={auprc:.4f}  (val n={len(va_idx)} pos={y_va.sum()})")

    assert np.isfinite(oof).all(), "OOF contains NaNs — check splits/training."
    return oof, fold_ids, fold_metrics


### 4) Models
We include:
- **Logistic Regression** baseline (interpretable)
- **HistGradientBoosting** (strong non-linear baseline, fast, no GPU)

You can swap HGB later for LightGBM/CatBoost if you decide to push peak performance.


In [ ]:
RANDOM_SEED = 42
N_SPLITS = 5

# Logistic Regression pipeline
lr = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False)),  # safe for sparse-ish wide data
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="lbfgs"
    )),
])

# HistGradientBoosting
hgb = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_depth=6,
        max_iter=400,
        min_samples_leaf=50,
        l2_regularization=0.0,
        random_state=RANDOM_SEED
    )),
])

models = [
    ("logreg", lr),
    ("hgb", hgb),
]


### 5) Run grouped CV + produce OOF predictions
We will save OOF predictions for **both** models, then choose the best for Notebook 04.


In [ ]:
all_metrics = []
oof_frames = []

for name, model in models:
    oof, fold_ids, metrics = run_group_cv_oof(model, X, y, groups, n_splits=N_SPLITS, model_name=name)
    all_metrics.extend(metrics)

    oof_df = df[["patient_id", "ICULOS"]].copy()
    oof_df["y"] = y
    oof_df["p_oof"] = oof
    oof_df["fold_id"] = fold_ids
    oof_df["model_name"] = name
    oof_frames.append(oof_df)

metrics_df = pd.DataFrame(all_metrics)
metrics_df


In [ ]:
# Summarize performance
summary = (metrics_df.groupby("model")[["auroc", "auprc"]]
           .agg(["mean", "std"])
          )
summary


### 6) Save artifacts
Notebook 04 will load `oof_predictions.parquet` and work only with these OOF probabilities.


In [ ]:
oof_all = pd.concat(oof_frames, ignore_index=True)
oof_all.to_parquet(OUT_OOF, index=False)

# metrics json
out = {
    "n_rows": int(len(df)),
    "n_patients": int(df["patient_id"].nunique()),
    "target_prevalence_row": float(df["y_within_H"].mean()),
    "n_splits": int(N_SPLITS),
    "metrics_per_fold": all_metrics,
    "summary": {
        m: {
            "auroc_mean": float(metrics_df.loc[metrics_df["model"]==m, "auroc"].mean()),
            "auroc_std":  float(metrics_df.loc[metrics_df["model"]==m, "auroc"].std()),
            "auprc_mean": float(metrics_df.loc[metrics_df["model"]==m, "auprc"].mean()),
            "auprc_std":  float(metrics_df.loc[metrics_df["model"]==m, "auprc"].std()),
        }
        for m in metrics_df["model"].unique()
    }
}
OUT_METRICS.write_text(json.dumps(out, indent=2))

print("Saved:")
print(" -", OUT_OOF)
print(" -", OUT_METRICS)


### 7) Choose the model for Notebook 04
Set `BEST_MODEL_NAME` to the model you will calibrate + use in the alert policy.

Tip: Often the best choice is the model with the best **AUPRC** (since positives are rare).


In [ ]:
BEST_MODEL_NAME = "hgb"  # change to "logreg" if it wins on your data

best_oof = oof_all[oof_all["model_name"] == BEST_MODEL_NAME].copy()
print(best_oof.head())
print("Best model rows:", len(best_oof))
print("Best model OOF AUROC:", roc_auc_score(best_oof["y"], best_oof["p_oof"]))
print("Best model OOF AUPRC:", average_precision_score(best_oof["y"], best_oof["p_oof"]))
